# CSE 25 – Introduction to Artificial Intelligence  
## Worksheet 14: Reinforcement Learning and Q-Learning

**Today’s focus:**  
How can an agent learn to make good decisions through trial and error, using rewards and punishments?

### Guiding Questions
1. What is reinforcement learning and how does it differ from supervised learning?
2. What are the key components: agent, environment, state, action, reward?
3. What is a Markov Decision Process (MDP)?
4. How does Q-learning work and what is the Q-update rule?
5. What is the exploration vs. exploitation dilemma?

### Learning Objectives
By the end of this worksheet, you will be able to:
- Describe the agent-environment interaction and identify its key components: states, actions, and rewards.
- Explain why a constant learning rate ($\alpha$) is necessary for tracking changing trends compared to a standard average ($1/t$).
- Deconstruct the Q-learning update rule into its constituent parts: Old Guess, Reward, and Future Potential.
- Calculate discounted future rewards using the discount factor ($\gamma$).
- Apply the $\epsilon$-greedy strategy to balance the exploration-exploitation dilemma.
- Perform policy extraction to derive optimal actions from a completed Q-table.

**Instructions:**  
Create a copy of this notebook and complete it during class.  
Work through the cells below **in order**.

You may discuss with your neighbors, but make sure you understand  
what each step is doing and why.

**Submission**  
When finished, download the notebook as a PDF and upload it to Gradescope under `In-Class – Week 9 Tuesday`.

To download as a PDF on DataHub:  
`File -> Save and Export Notebook As -> PDF`

### Reinforcement Learning

Reinforcement Learning is about learning by *trial and error*.

Imagine learning to play a new game:

- You try something.
- You see what happens.
- If it works, you’re more likely to do it again.
- If it doesn’t, you try something different.

That is the core idea of reinforcement learning.

Instead of being told the correct answer (like in supervised learning),
an agent must *interact* with its environment and learn from the outcomes
of its own actions.

The goal is simple:

> Learn to make decisions that lead to good outcomes over time.

#### Agent and Environment

In reinforcement learning, we describe the world using two parts:

- The **agent**: the learner and decision-maker.
- The **environment**: everything the agent interacts with.

NOTE: Agents are actors of any kind -- they can be embodied in the physical world or embedded in a virtual environment.

<img src="images/agent-env-mdp.png" width="500">

#### The Agent–Environment Interaction

Reinforcement learning is built around repeated interaction between the **agent** and the **environment**.

At each step:

1. The agent observes the current **state** $s$ of the environment.
2. The agent chooses an **action** $a$ from allowed actions.
3. The environment 
   - responds with a **reward** $r$
   - moves to the **next state** $s'$


#### Key Components

- **State ($s$)**: A representation of the environment.
- **Action ($a$)**: A choice the agent can make in that state to affect the environment.
- **Reward ($r$)**: A numerical signal from the environment indicating how good (or bad) the outcome was.
- **Next state ($s'$)**: The updated representation of the environment that results after taking action $a$ in state $s$.

### In-class Demo - Playing Nim

Nim is a turn-based game where players remove objects from a pile, and depending on the version, the player who takes the last object either wins (standard Nim) or loses (misère Nim).

#### The Game Rules (Our Version)
- We start with one pile with **6 objects**.
- On each turn, a player may remove **1 or 2 objects**.
- The player who takes the last object **loses**(misère Nim).

#### States, Actions and Rewards for our version of Nim

On a sheet of paper, we create 6 boxes labeled:

1, 2, 3, 4, 5, 6

Each box represents a *state* (the number of objects remaining).

Inside each box are two slips (denoting the *action*):
- "1" (remove 1 object)
- "2" (remove 2 objects)

**Reward ($r$)**:
  - $+1$ for a win,
  - $-1$ for a loss,
  - $0$ at all intermediate steps.

#### Training the agent

The agent will always play first.

To take an action: 
- Go to the box corresponding to the current state (the number of objects remaining).
- Randomly draw one slip.
- Take that action (if legal).

During the game:
- After drawing the slip, open it and leave it next to that state. Pick that number of objects from the pile.


After the game ends:

- If the agent **wins**:
  - Return all drawn slips to their boxes.
- If the agent **loses**:
  - Discard the slip corresponding to the agent's last action unless it's the only remaining action.
  - If removing that slip would leave a box empty:
    - Keep its last slip.
    - Instead discard the slip from the previous agent action.
  - Return all earlier slips to their boxes.

The hope is, that over time, losing actions disappear and the strategy improves!



##### Fun Fact: The Origin of "Matchbox Learning"

The [Nim activity](https://www.i-am.ai/build-your-own-ai.html) we just did is a simplified version of **MENACE** (*Machine Educable Noughts And Crosses Engine*), created by Donald Michie in 1961.

Before powerful digital computers were widely available, Michie used **matchboxes** filled with colored beads to "train" a system to play Tic-Tac-Toe using reinforcement principles:

- Each matchbox represented a board state.
- Each bead color represented a legal move from that state.
- After a win, several beads for the chosen moves were added (strong reward).
- After a loss, beads for the chosen moves were removed (punishment).
- After a draw, a small reward was given (fewer beads added than for a win).

Over time, the probability of selecting a move became proportional to the number of beads for that move, effectively learning a winning stratergy.

##### Explore the History:
* **The Original Paper:** [Experiments on the Mechanization of Game-Learning](https://people.csail.mit.edu/brooks/idocs/matchbox.pdf)
* **Watch it in Action:** [Playing Tic-Tac-Toe with Matchboxes](https://youtu.be/R9c-_neaxeU?si=WMZ-CxfG6mCqz6aX) (Modern demonstration of MENACE).

### Markov Decision Process (MDP)

In the Nim activity, we defined:

- States (number of objects remaining)
- Actions (remove 1 or 2 objects)
- Rewards (win or loss)
- Repeated interaction over time

To describe this more precisely, we use a mathematical framework called a **Markov Decision Process (MDP)**.


An MDP is a mathematical model for sequential decision-making when outcomes are *uncertain*.

It consists of:

- A set of **states** $S$ called the *state space*.
- A set of **actions** $A$ called the *action space*. 
- **Transition probabilities** $P(s' \mid s, a)$ which describe how likely we are to move to state $s'$ after taking action $a$ in state $s$. 
- A **reward function** $R(s, s', a)$ which gives the immediate reward received when the agent takes action $a$ in state $s$ and transitions to state $s'$. Rewards are **scalar values**. Higher values usually indicate better outcomes.

An MDP is a tuple of ($S$, $A$, $P(s' \mid s, a)$, $R(s, s', a)$)

#### Policy

A **policy** $\pi$ is a function that maps the state space $S$ to the action space $A$. It may be deterministic or stochastic.

#### Optimization Objective
**The goal** of reinforcement learning is to *learn a policy* that maximizes **total expected reward over time**.

**How Do We Actually Learn a Policy?**

In the Nim activity, we improved the policy by removing losing actions. That was an intuitive learning rule. But in general, we need something systematic. There are many reinforcement learning algorithms. In this course, we will study one such algorithm called **Q-learning.**

### Tabular Q-Learning 

In Q-learning, instead of slips, we store Q-values denoted as $Q(s, a)$ in a table. This value represents the **total expected reward** for taking action $a$ in state $s$ and then continuing to make the **best possible decisions** until the task is complete.

It essentially converts a short-term move into a long-term "quality" score, where higher values indicate a more successful path to the goal.

| State $s$| Take 1 | Take 2 |
|---|---|---|
| 1 | $Q(1,1)$ | $Q(1,2)$ |
| 2 | $Q(2,1)$ | $Q(2,2)$ |
| 3 | $Q(3,1)$ | $Q(3,2)$ |
| 4 | $Q(4,1)$ | $Q(4,2)$ |
| 5 | $Q(5,1)$ | $Q(5,2)$ |
| 6 | $Q(6,1)$ | $Q(6,2)$ |

In [ ]:
# We can represent the Q-table as a flat dictionary with tuple keys: (state, action)
# States: 1..6 (Playable states), Actions: 1 or 2

Q = {(state, action): 0.0 for state in range(1, 7) for action in (1, 2)}

def display_q_table(Q):
    # We only care about states where a move is possible (1-6)
    states = sorted({state for state, _ in Q if state > 0})

    print(f"{'State':>5} {'Take 1':>10} {'Take 2':>10}")
    print("-" * 27)

    for state in states:
        q_take_1 = Q[(state, 1)]
        q_take_2 = Q[(state, 2)]
        print(f"{state:>5} {q_take_1:>10.3f} {q_take_2:>10.3f}")

display_q_table(Q)

Now the question is, **how do we compute these Q-values from interaction with the environment?** For that we need to look 


Let's say I'm playing a game where I can score between 1 and 10 points. What would you predict my expected score would be the next time I play it?

What if you knew that in the past, I have scored these scores (not necessarily in this order): **3, 5, 3, 8, 10**

**Q. What score would you predict I will get?**

`YOUR ANSWER HERE`

*Hint: If you don't know the future, the best predictor of the next sample is usually the **mean** (average) of the previous samples.*

In [ ]:
scores = [3, 5, 3, 8, 10]

# Compute the mean of scores by adding all scores and dividing by the number of scores
mean = 0.0
# YOUR CODE HERE

print(f"Mean: {mean:.2f}")

#### Taking Averages, Sample by Sample

Now let's say we don't have all the scores at once, but we get one score at a time. How would we compute the average?  We can take an iterative approach. 

Instead of recalculating the entire sum each time, we update our estimate by weighting the **old average** and the **new score**:

At t=1, **Score 3**, Avg: **3**

At t=2, **Score 5**, Avg: $(\frac{1}{2})3 + (\frac{1}{2})5 = \mathbf{4}$

At t=3, **Score 3**, Avg: $(\frac{2}{3})4 + (\frac{1}{3})3 = {(1 - \frac{1}{3})4 + (\frac{1}{3})3} = \mathbf{\frac{11}{3}}$

At t=4, **Score 8**, Avg: $(\frac{3}{4})(\frac{11}{3}) + (\frac{1}{4})8 = {(1 - \frac{1}{4})(\frac{11}{3}) + (\frac{1}{4})8} = \mathbf{\frac{19}{4}}$


At t=5, **Score 10**, Avg: $(\frac{4}{5})(\frac{19}{4}) + (\frac{1}{5})10 = {(1 - \frac{1}{5})(\frac{19}{4}) + (\frac{1}{5})10} = \mathbf{\frac{29}{5} = 5.8}$


**Generalizing the Pattern**

From these steps, we can see that the new average ($\mu_t$) is always the old average weighted by the share of previous samples to the total, plus the new value weighted by its share of the total:

$$\mu_t =(\frac{t - 1}{t})\mu_{t-1} + \frac{1}{t}x_t$$ $$= (1 - \frac{1}{t})\mu_{t-1} + \frac{1}{t}x_t$$

Rearranging the terms, we get: 
$$\mu_t = \mu_{t-1} + \frac{1}{t}(x_t - \mu_{t-1}) $$

In [ ]:
iter_mean = 0.0

# Compute the mean of scores by iterating over the scores and updating the mean iteratively using the formula:
# iter_mean = iter_mean + (1 / t) * (x_t - iter_mean

for idx, x_t in enumerate(scores):
    # YOUR CODE HERE
    iter_mean = 0.0

print(iter_mean) # Should be the same as mean in the previous cell.

Okay, now let's say I'm playing the same game where I can score between 1 and 10 points. What would you predict my score would be the next time I play it? 

What if you knew that in the past, I have scored these scores in *this order*: 3, 3, 5, 8, 10

**Q. Is your guess about my next score higher, lower or the same as last time (5.8)?**

`YOUR ANSWER HERE`

We look at those numbers and we see a **trend**: I am getting better at the game! The scores of `3` from when I first started are now "stale" data -- they no longer represent my current skill level.

####  Tracking Change with a Learning Rate

**The Problem with $1/t$**

As you play more games, $t$ gets larger, which makes $1/t$ smaller. 
* By your 100th game, your latest score only changes your estimate by **1%**. 
* The formula becomes "heavy" with the past. It stays anchored to your beginner scores ($3, 3...$) and ignores your recent improvements ($10, 10...$).

**The Solution: Constant $\alpha$**

By replacing $1/t$ with a constant **learning rate** ($\alpha$), we ensure that the most recent score always has a fixed amount of influence (e.g., $10\%$). This allows our estimate to "track" changes in performance as they happen, rather than getting stuck in the past.

If we replace $\frac{1}{t}$ with a general **learning rate** $\alpha_t$, we get:

$$\mu_t = \mu_{t-1} + \alpha_t(x_t - \mu_{t-1})$$

The corrective term $x_t-\mu_{t-1}$ is known as a *temporal difference*. It represents the "error" -- the difference between the new sample and our current guess.

In [ ]:
scores_ordered = [3, 3, 5, 8, 10]
iter_mean_ordered = 0.0
alpha = 0.1

for i in range(20):
    for idx, x_t in enumerate(scores_ordered):
        # Compute the iterative mean with a learning rate alpha
        iter_mean_ordered = 0 # YOUR CODE HERE

        # print(f"Iteration {i+1}, Score {x_t}, Iterative Mean: {iter_mean_ordered:.2f}")

print(iter_mean_ordered)

Q. In the code above, why did we need the outer loop?

`YOUR ANSWER HERE`

Q. What happens is $\alpha = 1$?

`YOUR ASNWER HERE`

Q. What happens if it's very small, $\alpha = 0.001$?

`YOUR ANSWER HERE`

We can use this temporal difference update to update our Q-values as we interact with the environment and get rewards. 

We do not want to completely replace $Q(s,a)$. Instead, we move it slightly toward the new estimate:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha ( x_t - Q(s,a) )
$$

where $\alpha$ is the learning rate.


Now the question is: **What is $x_t$, also known as the TD-target, for Q-learning?**

In our previous example, $x_t$ was just a static score. In Q-Learning, our "sample" $x_t$ must account for both immediate rewards and future potential. 

After taking action $a$ in state $s$, we observe:
- An immediate reward **$r$**
- The next state **$s'$**

An action might not give a reward immediately, but it might lead to a good(or a bad) position. So, our target $x_t$ is the **reward now plus how good the next state looks**:

$$x_t = r + \max_{a'} Q(s',a')$$


**Adding the Final Piece: The Discount Factor ($\gamma$)**

If we simply added the future values directly, our Q-values could grow infinitely large. We also want to give our agent a sense of **urgency** - a reward of $+1$ right now should be worth more than a reward of $+1$ ten moves from now.

To handle this, we introduce the **Discount Factor ($\gamma$)**, a value between $0$ and $1$. 

Our final, polished target ($x_t$) becomes:
$$x_t = r + \gamma \max_{a'} Q(s',a')$$

#### The Full Q-Update Equation

Plugging our discounted target back into the update formula, we get the complete rule for training our AI:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big( \underbrace{[r + \gamma \max_{a'} Q(s',a')]}_{\text{New Evidence}} - \underbrace{Q(s,a)}_{\text{Old Guess}} \Big)$$

#### The Decision Dilemma: Exploration vs. Exploitation

Now that we have a mathematical way to update our "memory" ($Q$-values), how does the agent actually pick a move during the game? 

It faces a classic problem called the **Exploration vs. Exploitation Dilemma**:

* **Exploitation**: Choosing the move that currently has the highest score in our table. (Like going to your favorite restaurant because you *know* it's good).
* **Exploration**: Trying a random move to see if it leads to an even better outcome. (Like trying a brand new restaurant you've never been to).



If our agent only ever "exploits" early on, it might get stuck in a mediocre strategy just because it worked once. To find the truly optimal way to play, it must be curious!

#### The $\epsilon$-greedy Strategy

To balance these two needs, we use the **$\epsilon$-greedy strategy**. We define a parameter $\epsilon \in [0, 1]$ which determines the probability of exploration.

**The Decision Logic:**
1. Generate a random number (probability) $p \in [0, 1]$.
2. **If** $p < \epsilon$:
    - **Explore**: Select an action $a$ uniformly at random from the set of all legal actions.
3. **Else**:
    - **Exploit**: Select the action $a$ that maximizes the current Q-value: $a = \argmax_{a'} Q(s, a')$.

In [ ]:
import random
# To generate random numbers between 0 and 1 we can use the random.random().

print(random.random())

# We can use this to implement an epsilon-greedy policy for action selection.

In [ ]:
# If random.random() < epsilon, we choose a random action (exploration), 
# otherwise we choose the action with the highest Q-value (exploitation).

epsilon = 0.1
exploration = 0
exploitation = 0

for _ in range(5000):
    if random.random() < epsilon:
        exploration += 1
    else:
        exploitation += 1

print(f"Exploration count: {exploration}, Exploitation count: {exploitation}")
print(f"Exploration percentage: {exploration / 5000 * 100:.2f}%, Exploitation percentage: {exploitation / 5000 * 100:.2f}%")

Q. What happens if we set $\epsilon$ to 1?

`YOUR ANSWER HERE`

Q. What happens if we set $\epsilon$ to 0?

`YOUR ANSWER HERE`

### The Q-Learning Algorithm

Now that we have the update equation and a strategy for selecting actions, we can combine them into a complete learning loop. The goal of this algorithm is to iteratively improve our Q-table until the values accurately represent the quality of every possible move.

**Algorithm Logic:**

1.  **Initialize**: Set all $Q(s, a)$ values to 0 (the agent starts with no knowledge).
2.  **Repeat for each episode**:
    * Set the starting state $s$ (e.g., $s = 6$ in Nim).
    * **For each step in the episode**:
        * **Select** an action $a$ to apply in state $s$ using the **$\epsilon$-greedy** strategy.
        * **Execute** action $a$ and observe the outcome: the **reward** $r$ and the **next state** $s'$.
        * **Calculate the TD Error ($\delta$)**: 
            $$\delta \leftarrow r + \gamma \max_{a'} Q(s',a') - Q(s,a)$$
        * **Update the Table**: Apply the nudge to the memory:
            $$Q(s,a) \leftarrow Q(s,a) + \alpha \cdot \delta$$
        * **Transition**: Update our current position to the next state: $s \leftarrow s'$.
    * **Stop** when $s$ is a terminal state (the game is over).
3.  **End** training once the Q-values converge (they stop changing significantly).

#### Policy Extraction

After the training loop finishes, we have an "optimal" Q-table. However, a table of numbers isn't a strategy yet. We need to perform **Policy Extraction** to get our final rules for playing the game perfectly.

A **Policy ($\pi$)** is a mapping from states to actions. We extract the optimal policy by simply choosing the action that has the highest Q-value for every state:

$$\pi(s) = \argmax_{a} Q(s,a)$$

In [ ]:
# Q-table as a flat dictionary with tuple keys: (state, action)
# States: 0..6, Actions: 1 or 2
random.seed(42)  # For reproducibility
# Initializing Q-table with random values for demonstration
Q = {(state, action): random.random() for state in range(1, 7) for action in (1, 2)}

def display_q_table(Q):
    states = sorted({state for state, _ in Q})

    print(f"{'State':>5} {'Take 1':>10} {'Take 2':>10}")
    print("-" * 27)

    for state in states:
        q_take_1 = Q[(state, 1)]
        q_take_2 = Q[(state, 2)]
        print(f"{state:>5} {q_take_1:>10.3f} {q_take_2:>10.3f}")

display_q_table(Q)

In [ ]:
# Optimal Policy based on Q-table - argmax action for each state
optimal_policy = {}
for state in range(1,7):
    q_take_1 = Q[(state, 1)]
    q_take_2 = Q[(state, 2)]
    optimal_policy[state] = 1 if q_take_1 > q_take_2 else 2

print("Optimal Policy (state: best action):")
for state, action in optimal_policy.items():
    print(f"State {state}: Action {action}")
